In [1]:
from pyspark.sql.functions import *
from pyspark.sql import Row

StatementMeta(, dc586bf9-b9f3-4a69-9cd0-1e33d50ae2de, 3, Finished, Available, Finished, False)

**Date Dimension Table**

In [2]:
df = spark.read.table("anurag_silver_trips_clean")
# display(df)

StatementMeta(, dc586bf9-b9f3-4a69-9cd0-1e33d50ae2de, 4, Finished, Available, Finished, False)

In [3]:
min_date = df.select(min("PickUpDate")).collect()[0][0]
max_date = df.select(max("DropOffDate")).collect()[0][0]
display(min_date)
display(max_date)

date_df = spark.sql(f"""
SELECT explode(
    sequence(
        to_date('{min_date}'),
        to_date('{max_date}'),
        interval 1 day
    )
) AS Date
""")

date_df = date_df.withColumn("Year", year(col("Date"))) \
                 .withColumn("Month", month(col("Date"))) \
                 .withColumn("MonthName", date_format(col("Date"), "MMMM")) \
                 .withColumn("Day", dayofmonth(col("Date"))) \
                 .withColumn("DayName", date_format(col("Date"), "EEEE")) \
                 .withColumn("Quarter", quarter(col("Date"))) \
                 .withColumn("WeekOfYear", weekofyear(col("Date"))) \
                 .withColumn(

                     "DayNumber",

                     when(col("DayName") == "Sunday", 1)
                     .when(col("DayName") == "Monday", 2)
                     .when(col("DayName") == "Tuesday", 3)
                     .when(col("DayName") == "Wednesday", 4)
                     .when(col("DayName") == "Thursday", 5)
                     .when(col("DayName") == "Friday", 6)
                     .when(col("DayName") == "Saturday", 7)

                 )

#display(date_df)

StatementMeta(, dc586bf9-b9f3-4a69-9cd0-1e33d50ae2de, 5, Finished, Available, Finished, False)

datetime.date(2008, 12, 31)

datetime.date(2019, 1, 1)

**Saving Date Dimension Table**

In [4]:
date_df.write.mode("overwrite").saveAsTable(
    "anurag_gold_dim_date"
)

StatementMeta(, dc586bf9-b9f3-4a69-9cd0-1e33d50ae2de, 6, Finished, Available, Finished, False)

In [5]:
#date_df.printSchema()

StatementMeta(, dc586bf9-b9f3-4a69-9cd0-1e33d50ae2de, 7, Finished, Available, Finished, False)

**Hour and Minute Dimension Table**

In [6]:
time_df = spark.sql("""

SELECT
    hour,
    minute,
    second

FROM (

    SELECT explode(sequence(0,23)) AS hour

) h

CROSS JOIN (

    SELECT explode(sequence(0,59)) AS minute

) m

CROSS JOIN (

    SELECT explode(sequence(0,59)) AS second

) s

""")

# ------------------------------------------
# CREATE HH:MM:SS COLUMN
# ------------------------------------------

time_df = time_df.withColumn(

    "Time",

    format_string(
        "%02d:%02d:%02d",
        col("hour"),
        col("minute"),
        col("second")
    )

)

# ------------------------------------------
# DISPLAY
# ------------------------------------------

#display(time_df)

StatementMeta(, dc586bf9-b9f3-4a69-9cd0-1e33d50ae2de, 8, Finished, Available, Finished, False)

**Saving Hour and Minute Dimension Table**

In [7]:
time_df.write.mode("overwrite").saveAsTable(
    "anurag_gold_dim_time"
)



StatementMeta(, dc586bf9-b9f3-4a69-9cd0-1e33d50ae2de, 9, Finished, Available, Finished, False)

In [8]:
#time_df.printSchema()

StatementMeta(, dc586bf9-b9f3-4a69-9cd0-1e33d50ae2de, 10, Finished, Available, Finished, False)

**rateCodeID Column Dimension Table**

In [9]:


ratecode_data = [
    Row(rateCodeID=1, RateCodeName="Standard rate"),
    Row(rateCodeID=2, RateCodeName="JFK"),
    Row(rateCodeID=3, RateCodeName="Newark"),
    Row(rateCodeID=4, RateCodeName="Nassau/Westchester"),
    Row(rateCodeID=5, RateCodeName="Negotiated fare"),
    Row(rateCodeID=6, RateCodeName="Group ride")
]

dim_ratecode = spark.createDataFrame(ratecode_data)

#display(dim_ratecode)

StatementMeta(, dc586bf9-b9f3-4a69-9cd0-1e33d50ae2de, 11, Finished, Available, Finished, False)

In [10]:
dim_ratecode.write.mode("overwrite").saveAsTable(
    "anurag_gold_dim_ratecode"
)

StatementMeta(, dc586bf9-b9f3-4a69-9cd0-1e33d50ae2de, 12, Finished, Available, Finished, False)

In [11]:
#dim_ratecode.printSchema()

StatementMeta(, dc586bf9-b9f3-4a69-9cd0-1e33d50ae2de, 13, Finished, Available, Finished, False)

**StoreAndFwdFlag Dimesion Table**

In [12]:
storeforward_data = [

    Row(
        StoreForwardID = 1,
        StoreAndFwdFlag = "Y",
        Description = "Store and forward trip"
    ),

    Row(
        StoreForwardID = 0,
        StoreAndFwdFlag = "N",
        Description = "Not a store and forward trip"
    )

]

dim_storeforward = spark.createDataFrame(storeforward_data)

#display(dim_storeforward)

StatementMeta(, dc586bf9-b9f3-4a69-9cd0-1e33d50ae2de, 14, Finished, Available, Finished, False)

In [13]:
dim_storeforward.write.mode("overwrite").saveAsTable(
    "anurag_gold_dim_storeforward"
)

StatementMeta(, dc586bf9-b9f3-4a69-9cd0-1e33d50ae2de, 15, Finished, Available, Finished, False)

In [14]:
#dim_storeforward.printSchema()

StatementMeta(, dc586bf9-b9f3-4a69-9cd0-1e33d50ae2de, 16, Finished, Available, Finished, False)

**paymentType Dimension Table**

In [15]:
payment_data = [

    Row(
        PaymentTypeID = 1,
        PaymentType = "Credit Card"
    ),

    Row(
        PaymentTypeID = 2,
        PaymentType = "Cash"
    ),

    Row(
        PaymentTypeID = 3,
        PaymentType = "No Charge"
    ),

    Row(
        PaymentTypeID = 4,
        PaymentType = "Dispute"
    ),

    Row(
        PaymentTypeID = 5,
        PaymentType = "Unknown"
    ),

    Row(
        PaymentTypeID = 6,
        PaymentType = "Voided Trip"
    )

]

dim_payment = spark.createDataFrame(payment_data)

#display(dim_payment)

StatementMeta(, dc586bf9-b9f3-4a69-9cd0-1e33d50ae2de, 17, Finished, Available, Finished, False)

In [16]:
dim_payment.write.mode("overwrite").saveAsTable(
    "anurag_gold_dim_paymenttype"
)

StatementMeta(, dc586bf9-b9f3-4a69-9cd0-1e33d50ae2de, 18, Finished, Available, Finished, False)

In [17]:
#dim_payment.printSchema()

StatementMeta(, dc586bf9-b9f3-4a69-9cd0-1e33d50ae2de, 19, Finished, Available, Finished, False)

**triptype Dimension Table**

In [18]:
triptype_data = [

    Row(
        TripTypeID = 1,
        TripType = "Street-hail"
    ),

    Row(
        TripTypeID = 2,
        TripType = "Dispatch"
    )

]

dim_triptype = spark.createDataFrame(triptype_data)

#display(dim_triptype)

StatementMeta(, dc586bf9-b9f3-4a69-9cd0-1e33d50ae2de, 20, Finished, Available, Finished, False)

In [19]:
dim_triptype.write.mode("overwrite").saveAsTable(
    "anurag_gold_dim_triptype"
)

StatementMeta(, dc586bf9-b9f3-4a69-9cd0-1e33d50ae2de, 21, Finished, Available, Finished, False)

In [20]:
dim_triptype.printSchema()

StatementMeta(, dc586bf9-b9f3-4a69-9cd0-1e33d50ae2de, 22, Finished, Available, Finished, False)

root
 |-- TripTypeID: long (nullable = true)
 |-- TripType: string (nullable = true)



**vendorId Dimesion Table**

In [21]:
vendor_data = [

    Row(
        VendorID = 1,
        VendorName = "Creative Mobile Technologies, LLC"
    ),

    Row(
        VendorID = 2,
        VendorName = "VeriFone Inc"
    )

]

dim_vendor = spark.createDataFrame(vendor_data)

#display(dim_vendor)

StatementMeta(, dc586bf9-b9f3-4a69-9cd0-1e33d50ae2de, 23, Finished, Available, Finished, False)

In [22]:
dim_vendor.write.mode("overwrite").saveAsTable(
    "anurag_gold_dim_vendor"
)

StatementMeta(, dc586bf9-b9f3-4a69-9cd0-1e33d50ae2de, 24, Finished, Available, Finished, False)

In [23]:
#dim_vector.printSchema()

StatementMeta(, dc586bf9-b9f3-4a69-9cd0-1e33d50ae2de, 25, Finished, Available, Finished, False)

In [24]:
#display(df)

StatementMeta(, dc586bf9-b9f3-4a69-9cd0-1e33d50ae2de, 26, Finished, Available, Finished, False)

**Fact Table**

In [25]:
fact_df = df.select(
    "vendorID",
    "passengerCount",
    "tripDistance",
    "rateCodeID",
    "storeAndFwdFlag",
    "paymentType",
    "fareAmount",
    "extra",
    "mtaTax",
    "improvementSurcharge",
    "tipAmount",
    "tollsAmount",
    "totalAmount",
    "tripType",
    "PickUpDate",
    "PickUpTime",
    "DropOffDate",
    "DropOffTime"

)

#display(fact_df)

StatementMeta(, dc586bf9-b9f3-4a69-9cd0-1e33d50ae2de, 27, Finished, Available, Finished, False)

In [26]:
fact_df.write.mode("overwrite").saveAsTable(
    "anurag_gold_fact_trips"
)

StatementMeta(, dc586bf9-b9f3-4a69-9cd0-1e33d50ae2de, 28, Finished, Available, Finished, False)

In [27]:
#fact_df.printSchema()

StatementMeta(, dc586bf9-b9f3-4a69-9cd0-1e33d50ae2de, 29, Finished, Available, Finished, False)